# **ML Project Structure & Best Practices**

---

## Why Structure Matters

ML projects rot faster than ordinary software: notebooks, throwaway scripts, configs, datasets, checkpoints, and logs accumulate until nobody can answer *“which code + which config + which data produced this model?”*

Good structure buys you:

* **Reproducibility** — the same command rebuilds the same result.
* **Separation of concerns** — data code, model code, and training loops evolve independently.
* **Stable imports** — paths work the same from the CLI, a notebook, or a cluster job.

## A `src/` Layout

Putting the package under `src/` prevents accidentally importing from the working directory and forces you to install the project (`pip install -e .`), which is what CI and the cluster will do.

```
myproject/
├─ pyproject.toml          # deps + package metadata
├─ README.md
├─ conf/                   # Hydra/YAML configs (not code)
│  ├─ config.yaml
│  ├─ model/resnet.yaml
│  └─ data/cifar10.yaml
├─ src/myproject/
│  ├─ __init__.py
│  ├─ data/               # datasets, transforms, dataloaders
│  ├─ models/             # nn.Module definitions
│  ├─ train.py            # training loop / entry point
│  └─ utils/              # seeds, logging, checkpoints
├─ scripts/                # one-off CLI tools
├─ tests/
└─ outputs/                # checkpoints, logs (gitignored)
```

Keep **data and model definitions out of the training loop**: `train.py` should *wire together* a dataset, a model, an optimizer, and a config — not define them inline.

## Configuration Management (Hydra / YAML)

Hard-coded hyperparameters are unreproducible and unsearchable. Externalize them into composable YAML and let **Hydra** assemble and override them from the CLI.

`conf/config.yaml`:

```yaml
defaults:
  - model: resnet
  - data: cifar10
seed: 42
train:
  epochs: 50
  lr: 3e-4
  batch_size: 128
```

`src/myproject/train.py`:

```python
import hydra
from omegaconf import DictConfig

@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def main(cfg: DictConfig):
    set_seed(cfg.seed)
    model = build_model(cfg.model)
    train(model, cfg.train)

if __name__ == "__main__":
    main()
```

Override anything without touching code, and Hydra snapshots the *resolved* config next to the run:

```bash
python -m myproject.train train.lr=1e-3 train.epochs=100 model=resnet
```

## Reproducibility & Seeds

Reproducibility means: same code + same config + same data + same seed → same result. Seed every RNG and (when you need exactness) disable nondeterministic CUDA kernels.

```python
import os, random, numpy as np, torch

def set_seed(seed: int, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True)   # raises on nondeterministic ops
        torch.backends.cudnn.benchmark = False
```

Also pin **versions** (`pyproject.toml` / lockfile) and **data** (DVC commit), and log the resolved config + git SHA with every run — a seed alone is not reproducibility.

## Checklist

* `src/` layout, installed with `pip install -e .`
* Configs in YAML/Hydra, never hard-coded
* Data, models, and training loop in separate modules
* `set_seed()` called once at startup; versions and data pinned
* Outputs (checkpoints/logs) gitignored; config + git SHA logged per run

## References

* Hydra: https://hydra.cc/docs/intro/
* OmegaConf: https://omegaconf.readthedocs.io
* `torch.use_deterministic_algorithms`: https://pytorch.org/docs/stable/notes/randomness.html
* Python `src` layout: https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/